# EoMT Model Comparison on Colab

This notebook compares `eomt_cityscapes.bin` and `eomt_coco.bin` under the three segmentation paradigms on shared known classes.

It includes:
- quantitative semantic evaluation on Cityscapes validation
- quantitative instance and panoptic evaluation on COCO panoptic validation
- qualitative side-by-side semantic, instance, and panoptic visualizations

The comparison is intentionally explicit about the limitation that the two checkpoints were trained on different datasets and task formulations.

## Colab Setup

Recommended flow:
1. Upload or clone this repository in Colab.
2. Put the datasets and checkpoints somewhere readable from Colab.
3. Update the paths in the configuration cell below.

Expected files:
- `trained_models/eomt_cityscapes.bin`
- `trained_models/eomt_coco.bin`
- Cityscapes zips: `leftImg8bit_trainvaltest.zip`, `gtFine_trainvaltest.zip`
- COCO zips: `val2017.zip`, `panoptic_annotations_trainval2017.zip`


In [ ]:
!pip install -q -r requirements.txt pycocotools

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

CITYSCAPES_DATA_PATH = Path('/content/data/cityscapes')
COCO_DATA_PATH = Path('/content/data/coco')

CITYSCAPES_CKPT = PROJECT_ROOT / 'trained_models' / 'eomt_cityscapes.bin'
COCO_CKPT = PROJECT_ROOT / 'trained_models' / 'eomt_coco.bin'

CITYSCAPES_CONFIG = PROJECT_ROOT / 'eomt' / 'configs' / 'dinov2' / 'cityscapes' / 'semantic' / 'eomt_base_640.yaml'
COCO_CONFIG = PROJECT_ROOT / 'eomt' / 'configs' / 'dinov2' / 'coco' / 'panoptic' / 'eomt_base_640_2x.yaml'

NUM_SEMANTIC_SAMPLES = 100
NUM_COCO_SAMPLES = 100
NUM_QUALITATIVE_SAMPLES = 4
INSTANCE_TOP_K = 100
DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'

OUTPUT_DIR = PROJECT_ROOT / 'comparison_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Device:', DEVICE)

In [ ]:
import inspect
import importlib
import json
import math
import sys
import warnings
import zipfile
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import yaml
from PIL import Image
from torchmetrics.detection import MeanAveragePrecision, PanopticQuality
from torchvision.datasets import Cityscapes

sys.path.insert(0, str(PROJECT_ROOT / 'eomt'))

from datasets.coco_panoptic import COCOPanoptic, CLASS_MAPPING as COCO_CLASS_MAPPING

warnings.filterwarnings('ignore', message=r'.*already saved during checkpointing.*')
torch.set_grad_enabled(False)
torch.manual_seed(0)

In [ ]:
CITYSCAPES_ID_TO_NAME = {cls.train_id: cls.name for cls in Cityscapes.classes if not cls.ignore_in_eval}
CITYSCAPES_NAME_TO_ID = {v: k for k, v in CITYSCAPES_ID_TO_NAME.items()}

COMMON_STUFF = ['road', 'sidewalk', 'building', 'sky']
COMMON_THINGS = ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck', 'train']
COMMON_CLASS_NAMES = COMMON_STUFF + COMMON_THINGS
COMMON_NAME_TO_ID = {name: idx for idx, name in enumerate(COMMON_CLASS_NAMES)}
COMMON_VOID_ID = len(COMMON_CLASS_NAMES)

CITYSCAPES_COMMON_MAP = {
    CITYSCAPES_NAME_TO_ID['road']: COMMON_NAME_TO_ID['road'],
    CITYSCAPES_NAME_TO_ID['sidewalk']: COMMON_NAME_TO_ID['sidewalk'],
    CITYSCAPES_NAME_TO_ID['building']: COMMON_NAME_TO_ID['building'],
    CITYSCAPES_NAME_TO_ID['sky']: COMMON_NAME_TO_ID['sky'],
    CITYSCAPES_NAME_TO_ID['person']: COMMON_NAME_TO_ID['person'],
    CITYSCAPES_NAME_TO_ID['bicycle']: COMMON_NAME_TO_ID['bicycle'],
    CITYSCAPES_NAME_TO_ID['car']: COMMON_NAME_TO_ID['car'],
    CITYSCAPES_NAME_TO_ID['motorcycle']: COMMON_NAME_TO_ID['motorcycle'],
    CITYSCAPES_NAME_TO_ID['bus']: COMMON_NAME_TO_ID['bus'],
    CITYSCAPES_NAME_TO_ID['truck']: COMMON_NAME_TO_ID['truck'],
    CITYSCAPES_NAME_TO_ID['train']: COMMON_NAME_TO_ID['train'],
}

CITYSCAPES_THING_IDS = {CITYSCAPES_NAME_TO_ID[name] for name in ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck', 'train']}
CITYSCAPES_STUFF_IDS = set(CITYSCAPES_COMMON_MAP) - CITYSCAPES_THING_IDS

COCO_ALIASES = {
    'road': ['road'],
    'sidewalk': ['sidewalk'],
    'building': ['building', 'building-other-merged'],
    'sky': ['sky', 'sky-other-merged'],
    'person': ['person'],
    'bicycle': ['bicycle'],
    'car': ['car'],
    'motorcycle': ['motorcycle'],
    'bus': ['bus'],
    'truck': ['truck'],
    'train': ['train'],
}

def load_yaml(path):
    with open(path, 'r', encoding='utf-8') as handle:
        return yaml.safe_load(handle)

def load_coco_categories(coco_data_path):
    zip_path = Path(coco_data_path) / 'panoptic_annotations_trainval2017.zip'
    with zipfile.ZipFile(zip_path) as zf:
        with zf.open('annotations/panoptic_val2017.json') as handle:
            meta = json.load(handle)
    return meta['categories']

def resolve_coco_common_map(coco_data_path):
    categories = load_coco_categories(coco_data_path)
    by_name = {cat['name']: cat['id'] for cat in categories}
    common_map = {}
    for common_name, aliases in COCO_ALIASES.items():
        original_id = next((by_name[name] for name in aliases if name in by_name), None)
        if original_id is None:
            print(f'Warning: could not resolve COCO class for {common_name}')
            continue
        if original_id not in COCO_CLASS_MAPPING:
            print(f'Warning: COCO class id {original_id} for {common_name} is not in continuous mapping')
            continue
        continuous_id = COCO_CLASS_MAPPING[original_id]
        common_map[continuous_id] = COMMON_NAME_TO_ID[common_name]
    return common_map

COCO_COMMON_MAP = resolve_coco_common_map(COCO_DATA_PATH)
COCO_THING_IDS = {model_id for model_id, common_id in COCO_COMMON_MAP.items() if COMMON_CLASS_NAMES[common_id] in COMMON_THINGS}
COCO_STUFF_IDS = {model_id for model_id, common_id in COCO_COMMON_MAP.items() if COMMON_CLASS_NAMES[common_id] in COMMON_STUFF}

def import_class(class_path):
    module_name, class_name = class_path.rsplit('.', 1)
    return getattr(importlib.import_module(module_name), class_name)

def build_data_module(config, data_path, batch_size=1):
    data_cls = import_class(config['data']['class_path'])
    kwargs = dict(config['data'].get('init_args', {}))
    kwargs.update({'path': str(data_path), 'batch_size': batch_size, 'num_workers': 0, 'check_empty_targets': False})
    return data_cls(**kwargs).setup()

def select_kwargs(callable_obj, base_kwargs):
    params = inspect.signature(callable_obj).parameters
    return {key: value for key, value in base_kwargs.items() if key in params and value is not None}

def build_model(config, data_module, ckpt_path):
    encoder_cfg = config['model']['init_args']['network']['init_args']['encoder']
    encoder_cls = import_class(encoder_cfg['class_path'])
    encoder_kwargs = dict(encoder_cfg.get('init_args', {}))
    if 'img_size' in inspect.signature(encoder_cls).parameters:
        encoder_kwargs.setdefault('img_size', data_module.img_size)
    encoder = encoder_cls(**encoder_kwargs)

    network_cfg = config['model']['init_args']['network']
    network_cls = import_class(network_cfg['class_path'])
    network_kwargs = dict(network_cfg.get('init_args', {}))
    network_kwargs.pop('encoder', None)
    network_kwargs.setdefault('encoder', encoder)
    network_kwargs.setdefault('num_classes', data_module.num_classes)
    network = network_cls(**network_kwargs)

    task_cfg = config['model']
    task_cls = import_class(task_cfg['class_path'])
    task_kwargs = dict(task_cfg.get('init_args', {}))
    task_kwargs['network'] = network
    task_kwargs.setdefault('img_size', data_module.img_size)
    task_kwargs.setdefault('num_classes', data_module.num_classes)
    if hasattr(data_module, 'stuff_classes'):
        task_kwargs.setdefault('stuff_classes', data_module.stuff_classes)
    task = task_cls(**select_kwargs(task_cls, task_kwargs))

    ckpt = task._load_ckpt(str(ckpt_path), load_ckpt_class_head=True)
    incompatible = task.load_state_dict(ckpt, strict=False)
    task._raise_on_incompatible(incompatible, load_ckpt_class_head=True)
    return task.to(DEVICE).eval()

def map_semantic_mask(mask, class_map, void_id=255):
    mapped = np.full(mask.shape, void_id, dtype=np.int64)
    for old_id, new_id in class_map.items():
        mapped[mask == old_id] = new_id
    return mapped

def update_confusion(confusion, pred, target, num_classes):
    valid = (target >= 0) & (target < num_classes)
    pred = pred[valid]
    target = target[valid]
    counts = np.bincount(num_classes * target + pred, minlength=num_classes ** 2)
    confusion += counts.reshape(num_classes, num_classes)

def compute_iou_table(confusion):
    rows = []
    ious = []
    for idx, name in enumerate(COMMON_CLASS_NAMES):
        tp = confusion[idx, idx]
        fp = confusion[:, idx].sum() - tp
        fn = confusion[idx, :].sum() - tp
        denom = tp + fp + fn
        iou = float(tp / denom) if denom > 0 else float('nan')
        rows.append({'class': name, 'iou': iou})
        if not math.isnan(iou):
            ious.append(iou)
    return rows, float(np.mean(ious)) if ious else float('nan')

def panoptic_target_from_masks(target, class_map):
    h, w = target['masks'].shape[-2:]
    panoptic = torch.full((h, w, 2), 0, dtype=torch.long)
    panoptic[:, :, 0] = COMMON_VOID_ID
    segment_id = 1
    for idx, label in enumerate(target['labels'].tolist()):
        if label not in class_map:
            continue
        mask = target['masks'][idx]
        panoptic[:, :, 0][mask] = class_map[label]
        panoptic[:, :, 1][mask] = segment_id
        segment_id += 1
    return panoptic

def remap_panoptic_prediction(pred, class_map):
    out = pred.clone().long()
    classes = out[:, :, 0]
    mapped = torch.full_like(classes, COMMON_VOID_ID)
    for old_id, new_id in class_map.items():
        mapped[classes == old_id] = new_id
    out[:, :, 0] = mapped
    out[:, :, 1][mapped == COMMON_VOID_ID] = 0
    return out

def rgb_from_tensor(img):
    arr = img.detach().cpu().numpy().transpose(1, 2, 0).astype(np.uint8)
    return arr

def colorize_semantic(mask):
    cmap = plt.get_cmap('tab20', len(COMMON_CLASS_NAMES) + 1)
    return cmap(mask.clip(0, len(COMMON_CLASS_NAMES)))[:, :, :3]

def overlay_instances(img, pred, model_class_names):
    canvas = img.astype(np.float32) / 255.0
    colors = plt.get_cmap('tab10', max(1, len(pred['masks'])))
    for idx, mask in enumerate(pred['masks'][:10]):
        color = np.array(colors(idx)[:3])
        mask_np = mask.detach().cpu().numpy().astype(bool)
        canvas[mask_np] = 0.45 * canvas[mask_np] + 0.55 * color
    return canvas

def latest_layer(outputs):
    mask_logits_per_layer, class_logits_per_layer = outputs
    return mask_logits_per_layer[-1], class_logits_per_layer[-1]


In [ ]:
city_config = load_yaml(CITYSCAPES_CONFIG)
coco_config = load_yaml(COCO_CONFIG)

city_data = build_data_module(city_config, CITYSCAPES_DATA_PATH, batch_size=1)
coco_data = build_data_module(coco_config, COCO_DATA_PATH, batch_size=1)

city_model = build_model(city_config, city_data, CITYSCAPES_CKPT)
coco_model = build_model(coco_config, coco_data, COCO_CKPT)

city_val_loader = city_data.val_dataloader()
coco_val_loader = coco_data.val_dataloader()

print('Cityscapes img_size:', city_data.img_size)
print('COCO img_size:', coco_data.img_size)

In [ ]:
def predict_semantic_ids(model, imgs, mode):
    img_sizes = [img.shape[-2:] for img in imgs]
    if mode == 'semantic':
        crops, origins = model.window_imgs_semantic(imgs)
        mask_logits, class_logits = latest_layer(model(crops.to(DEVICE)))
        mask_logits = F.interpolate(mask_logits, model.img_size, mode='bilinear')
        crop_logits = model.to_per_pixel_logits_semantic(mask_logits, class_logits)
        logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)
    else:
        transformed = model.resize_and_pad_imgs_instance_panoptic(imgs)
        mask_logits, class_logits = latest_layer(model(transformed.to(DEVICE)))
        mask_logits = F.interpolate(mask_logits, model.img_size, mode='bilinear')
        mask_logits = model.revert_resize_and_pad_logits_instance_panoptic(mask_logits, img_sizes)
        logits = model.to_per_pixel_logits_semantic(torch.stack(mask_logits), class_logits)
        logits = [x for x in logits]
    return [torch.argmax(logit, dim=0).cpu().numpy() for logit in logits]

def predict_instance(model, imgs, top_k, thing_ids, class_map):
    img_sizes = [img.shape[-2:] for img in imgs]
    transformed = model.resize_and_pad_imgs_instance_panoptic(imgs)
    mask_logits, class_logits = latest_layer(model(transformed.to(DEVICE)))
    mask_logits = F.interpolate(mask_logits, model.img_size, mode='bilinear')
    mask_logits = model.revert_resize_and_pad_logits_instance_panoptic(mask_logits, img_sizes)

    preds = []
    for batch_idx in range(len(mask_logits)):
        scores = class_logits[batch_idx].softmax(dim=-1)[:, :-1]
        labels = torch.arange(scores.shape[-1], device=scores.device).unsqueeze(0).repeat(scores.shape[0], 1).flatten(0, 1)
        topk_scores, topk_indices = scores.flatten(0, 1).topk(min(top_k, scores.numel()), sorted=False)
        labels = labels[topk_indices]
        query_indices = topk_indices // scores.shape[-1]
        selected_masks = mask_logits[batch_idx][query_indices]
        masks = selected_masks > 0
        mask_scores = (selected_masks.sigmoid().flatten(1) * masks.flatten(1)).sum(1) / (masks.flatten(1).sum(1) + 1e-6)
        scores_out = topk_scores * mask_scores

        keep = [(label.item() in thing_ids) and (label.item() in class_map) for label in labels]
        keep = torch.tensor(keep, dtype=torch.bool, device=labels.device)
        labels = labels[keep]
        masks = masks[keep]
        scores_out = scores_out[keep]
        mapped_labels = torch.tensor([class_map[label.item()] for label in labels], dtype=torch.long)
        preds.append({'masks': masks.cpu(), 'labels': mapped_labels.cpu(), 'scores': scores_out.cpu()})
    return preds

def predict_panoptic(model, imgs, stuff_ids, class_map):
    img_sizes = [img.shape[-2:] for img in imgs]
    transformed = model.resize_and_pad_imgs_instance_panoptic(imgs)
    mask_logits, class_logits = latest_layer(model(transformed.to(DEVICE)))
    mask_logits = F.interpolate(mask_logits, model.img_size, mode='bilinear')
    mask_logits = model.revert_resize_and_pad_logits_instance_panoptic(mask_logits, img_sizes)
    preds = model.to_per_pixel_preds_panoptic(mask_logits, class_logits, list(stuff_ids), mask_thresh=0.8, overlap_thresh=0.8)
    return [remap_panoptic_prediction(pred.cpu(), class_map) for pred in preds]

def prepare_coco_instance_targets(targets, class_map):
    prepared = []
    for target in targets:
        keep = [label.item() in class_map and COMMON_CLASS_NAMES[class_map[label.item()]] in COMMON_THINGS for label in target['labels']]
        keep = torch.tensor(keep, dtype=torch.bool)
        labels = target['labels'][keep]
        prepared.append({
            'masks': target['masks'][keep].cpu(),
            'labels': torch.tensor([class_map[label.item()] for label in labels], dtype=torch.long),
            'iscrowd': target['is_crowd'][keep].cpu(),
        })
    return prepared

In [ ]:
semantic_conf_city = np.zeros((len(COMMON_CLASS_NAMES), len(COMMON_CLASS_NAMES)), dtype=np.int64)
semantic_conf_coco = np.zeros((len(COMMON_CLASS_NAMES), len(COMMON_CLASS_NAMES)), dtype=np.int64)

for step, (imgs, targets) in enumerate(city_val_loader):
    if step >= NUM_SEMANTIC_SAMPLES:
        break
    gt = city_model.to_per_pixel_targets_semantic(targets, city_model.ignore_idx)[0].cpu().numpy()
    gt_common = map_semantic_mask(gt, CITYSCAPES_COMMON_MAP, void_id=255)

    city_pred = predict_semantic_ids(city_model, imgs, mode='semantic')[0]
    city_pred_common = map_semantic_mask(city_pred, CITYSCAPES_COMMON_MAP, void_id=255)

    coco_pred = predict_semantic_ids(coco_model, imgs, mode='panoptic')[0]
    coco_pred_common = map_semantic_mask(coco_pred, COCO_COMMON_MAP, void_id=255)

    update_confusion(semantic_conf_city, city_pred_common, gt_common, len(COMMON_CLASS_NAMES))
    update_confusion(semantic_conf_coco, coco_pred_common, gt_common, len(COMMON_CLASS_NAMES))

semantic_rows_city, semantic_miou_city = compute_iou_table(semantic_conf_city)
semantic_rows_coco, semantic_miou_coco = compute_iou_table(semantic_conf_coco)

print('Semantic mIoU on shared classes')
print('cityscapes checkpoint:', round(semantic_miou_city, 4))
print('coco checkpoint:', round(semantic_miou_coco, 4))
semantic_rows_city, semantic_rows_coco

In [ ]:
instance_metric_city = MeanAveragePrecision(iou_type='segm')
instance_metric_coco = MeanAveragePrecision(iou_type='segm')

for step, (imgs, targets) in enumerate(coco_val_loader):
    if step >= NUM_COCO_SAMPLES:
        break
    gt_targets = prepare_coco_instance_targets(targets, COCO_COMMON_MAP)
    city_preds = predict_instance(city_model, imgs, INSTANCE_TOP_K, CITYSCAPES_THING_IDS, CITYSCAPES_COMMON_MAP)
    coco_preds = predict_instance(coco_model, imgs, INSTANCE_TOP_K, COCO_THING_IDS, COCO_COMMON_MAP)
    instance_metric_city.update(city_preds, gt_targets)
    instance_metric_coco.update(coco_preds, gt_targets)

instance_results_city = {k: float(v) for k, v in instance_metric_city.compute().items() if isinstance(v, torch.Tensor)}
instance_results_coco = {k: float(v) for k, v in instance_metric_coco.compute().items() if isinstance(v, torch.Tensor)}

print('Instance segmentation on shared thing classes vs COCO panoptic val')
print('cityscapes checkpoint:', instance_results_city)
print('coco checkpoint:', instance_results_coco)

In [ ]:
panoptic_metric_city = PanopticQuality(list(range(len(COMMON_STUFF), len(COMMON_CLASS_NAMES))), list(range(len(COMMON_STUFF))) + [COMMON_VOID_ID], return_sq_and_rq=True, return_per_class=True)
panoptic_metric_coco = PanopticQuality(list(range(len(COMMON_STUFF), len(COMMON_CLASS_NAMES))), list(range(len(COMMON_STUFF))) + [COMMON_VOID_ID], return_sq_and_rq=True, return_per_class=True)

for step, (imgs, targets) in enumerate(coco_val_loader):
    if step >= NUM_COCO_SAMPLES:
        break
    gt_panoptic = torch.stack([panoptic_target_from_masks(target, COCO_COMMON_MAP) for target in targets])
    city_panoptic = torch.stack(predict_panoptic(city_model, imgs, CITYSCAPES_STUFF_IDS, CITYSCAPES_COMMON_MAP))
    coco_panoptic = torch.stack(predict_panoptic(coco_model, imgs, COCO_STUFF_IDS, COCO_COMMON_MAP))
    panoptic_metric_city.update(city_panoptic, gt_panoptic)
    panoptic_metric_coco.update(coco_panoptic, gt_panoptic)

panoptic_results_city = panoptic_metric_city.compute()[:-1]
panoptic_results_coco = panoptic_metric_coco.compute()[:-1]

print('Panoptic PQ / SQ / RQ per shared class')
print('cityscapes checkpoint mean:', panoptic_results_city.mean(0))
print('coco checkpoint mean:', panoptic_results_coco.mean(0))

In [ ]:
summary = {
    'semantic_cityscapes_miou': semantic_miou_city,
    'semantic_coco_miou': semantic_miou_coco,
    'instance_cityscapes': instance_results_city,
    'instance_coco': instance_results_coco,
    'panoptic_cityscapes_mean': panoptic_results_city.mean(0).tolist(),
    'panoptic_coco_mean': panoptic_results_coco.mean(0).tolist(),
}

with open(OUTPUT_DIR / 'quantitative_summary.json', 'w', encoding='utf-8') as handle:
    json.dump(summary, handle, indent=2)

summary

In [ ]:
for step, (imgs, targets) in enumerate(city_val_loader):
    if step >= NUM_QUALITATIVE_SAMPLES:
        break
    rgb = rgb_from_tensor(imgs[0])
    city_sem = map_semantic_mask(predict_semantic_ids(city_model, imgs, mode='semantic')[0], CITYSCAPES_COMMON_MAP, void_id=COMMON_VOID_ID)
    coco_sem = map_semantic_mask(predict_semantic_ids(coco_model, imgs, mode='panoptic')[0], COCO_COMMON_MAP, void_id=COMMON_VOID_ID)
    city_inst = predict_instance(city_model, imgs, INSTANCE_TOP_K, CITYSCAPES_THING_IDS, CITYSCAPES_COMMON_MAP)[0]
    coco_inst = predict_instance(coco_model, imgs, INSTANCE_TOP_K, COCO_THING_IDS, COCO_COMMON_MAP)[0]
    city_pan = predict_panoptic(city_model, imgs, CITYSCAPES_STUFF_IDS, CITYSCAPES_COMMON_MAP)[0][:, :, 0].numpy()
    coco_pan = predict_panoptic(coco_model, imgs, COCO_STUFF_IDS, COCO_COMMON_MAP)[0][:, :, 0].numpy()

    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    axes[0, 0].imshow(rgb)
    axes[0, 0].set_title('RGB')
    axes[0, 1].imshow(colorize_semantic(city_sem))
    axes[0, 1].set_title('Cityscapes semantic')
    axes[0, 2].imshow(colorize_semantic(coco_sem))
    axes[0, 2].set_title('COCO semantic')
    axes[0, 3].imshow(colorize_semantic(map_semantic_mask(city_model.to_per_pixel_targets_semantic(targets, city_model.ignore_idx)[0].cpu().numpy(), CITYSCAPES_COMMON_MAP, void_id=COMMON_VOID_ID)))
    axes[0, 3].set_title('GT shared semantic')
    axes[1, 0].imshow(overlay_instances(rgb, city_inst, COMMON_CLASS_NAMES))
    axes[1, 0].set_title('Cityscapes instance')
    axes[1, 1].imshow(overlay_instances(rgb, coco_inst, COMMON_CLASS_NAMES))
    axes[1, 1].set_title('COCO instance')
    axes[1, 2].imshow(colorize_semantic(city_pan))
    axes[1, 2].set_title('Cityscapes panoptic')
    axes[1, 3].imshow(colorize_semantic(coco_pan))
    axes[1, 3].set_title('COCO panoptic')
    for ax in axes.ravel():
        ax.axis('off')
    plt.tight_layout()
    out_path = OUTPUT_DIR / f'qualitative_{step:02d}.png'
    plt.savefig(out_path, dpi=160)
    plt.show()

## What to comment on in the report

Quantitatively:
- semantic `mIoU` and per-class IoU on the shared label set
- instance `mAP`, `mAP50`, `mAP75`
- panoptic `PQ`, `SQ`, `RQ`

Qualitatively:
- object separation in crowded scenes
- boundary quality on road and sidewalk
- failure modes on small or thin objects
- whether the COCO model is more object-centric and the Cityscapes model more layout-centric
- whether the semantic checkpoint collapses instances that the panoptic checkpoint separates